## Real Data in Databricks Free Edition

You can work with real data in Databricks Free Edition, but the options are more limited than in a paid workspace. The main realistic path is **file-based streaming**, not live external systems like Kafka.

### What Doesn't Work in Free Edition

Connecting to external streaming services like **Apache Kafka** is not supported in the serverless Free Edition. If you try, you will see errors like:

- `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`
- `TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED`

### What Does Work: File-Based Streaming

Use **Auto Loader** to stream data from files stored in **Unity Catalog Volumes**. This is the closest you can get to practicing with "real" streaming data, because new files that appear in a volume are picked up as new data.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.streaming_data;

In [0]:
#use spark.readStream to read from that volume using Auto Loader. The schemaLocation option is important for schema inference

# Define paths
source_path = "/Volumes/workspace/default/streaming_data"
checkpoint_path = "/Volumes/workspace/default/checkpoints/wikipedia_stream"

# Read the stream using Auto Loader
raw_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(source_path)
)

# Write to a Delta table
(raw_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.wikipedia_stream_output")
)